## PCM_pix — clean runner notebook

Этот ноутбук — **тонкий сценарий**:
- грузит данные из `data/`
- грузит/обучает суррогаты (legacy или new)
- оценивает качество суррогатов
- запускает оптимизацию (PSO / DE / PSO→DE)
- сохраняет результаты в `outputs/<run_name>/`

Старые большие ноутбуки остаются как архив (`3rd art_...`, `to_server_arch-...`).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

# --- CONFIG (меняй здесь) ---
CFG = {
    "run_name": "2026_03_12",
    
    # surrogate
    "ann_data_dir": "data/ann_models", #где лежат уже тренированные модели
    "adb_data_dir": "data",
    "adb_data":["Sb2Se3 - amorphous_mesh_sbse_2025.txt", "Sb2Se3 - crystal_mesh12_sbse_2025.txt",], #данные из люма
    "wl": 1.55e-6,  # фильтруем датасет по длинам волн
   
    "train_load_mode": "load", # train/load
    "device": "cpu", #cpu/cuda
    "epochs": 10000,
    "lr": 1e-3,
    "qa_n": 5000, # количество точек для оценки суррогатов

    "preset_dir": "data/preset/",

    # fabrication settings
    "Nn": 11,
    "b_min_m": 50e-9, #fabrication limit

    #optimization

    "opt_mode": "hybrid_pso_de",      # "preset" | "pso" | "pso_until" | "de_full" | "hybrid_pso_de"
    
    
    "preset_name": "pos_server_2026_03_04_night",

    # hyperopt modes
    # - legacy ключ (общий), оставлен для совместимости
    # - новые ключи позволяют управлять PSO и DE независимо
    #"hyperopt_mode": "run",        # legacy: "auto" | "use_saved" | "run"
    "pso_hyperopt_mode": "use_saved",    # "auto" | "use_saved" | "run"
    "de_hyperopt_mode": "use_saved",     # "auto" | "use_saved" | "run"

    # PSO
    "pso_threshold": 4,
    "pso_max_restarts": 20,
    "pso_n_particles": 3000,
    "pso_iters": 1000,
    "pso_c1": 0.5,
    "pso_c2": 0.3,
    "pso_w": 0.9,

    # DE (full compatibility)
    "de_init_mode": "init_ar",    # "init_ar" - через начальное состояние, которое генерируется из исходника с шумом 
    "de_init_spread": 0.1, # 10% от исходника
   #"de_init_mode": "x0",    # "x0" - через начальное состояние
    "de_init_N": 10000,
    "de_mutation": 0.99,
    "de_recombination": 0.1,
    "de_maxiter": 50000,
    "de_popsize": 500,
    "de_tol": 1e-8,
    "de_atol": 1e-3,
    "de_polish": True,
    "de_callback_every": 1000,
    "de_use_linear_constraint": False,


}

# optional: reproducibility seed
SEED = 1
np.random.seed(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from pcm_pix.run import init_notebook_run
run, logger = init_notebook_run(CFG, notebook_name="main")

2026-03-14 16:06:30,974 | INFO | run_dir=/media/slava/Data/git/PCM_pix/outputs/2026_03_12
2026-03-14 16:06:30,975 | INFO | main started


In [12]:
# --- Dataset + surrogates + Metrics ---
from pcm_pix.nn_surrogate import build_ANN
df, data_0, data_1, X_0, y_0, X_1, y_1, sur0, sur1, qa_am, qa_cr = build_ANN(CFG, run)
qa_am, qa_cr  # чтобы увидеть словари метрик в output

2026-03-14 16:06:33,847 | INFO | loading surrogates from outputs/2026_03_12/models
2026-03-14 16:06:33,857 | INFO | surrogates OK
2026-03-14 16:06:33,858 | INFO | dataset: df=98390 data_0=60360 data_1=38030
2026-03-14 16:06:33,859 | INFO | X_0=(60360, 3) y_0=(60360, 4) | X_1=(38030, 3) y_1=(38030, 4)
2026-03-14 16:06:33,953 | INFO | QA am: {'label': 'am', 'n': 5000, 'R_mae': 0.0025169088971111007, 'R_rmse': 0.005963946653039524, 'T_mae': 0.0031418092457471784, 'T_rmse': 0.007820708795250876, 'phiR_mae': 0.02438346642204953, 'phiR_rmse': 0.1074883793671806, 'phiT_mae': 0.014347813825024636, 'phiT_rmse': 0.11495134450534344, 'R_out_of_01': 15, 'T_out_of_01': 2}
2026-03-14 16:06:33,954 | INFO | QA cr: {'label': 'cr', 'n': 5000, 'R_mae': 0.00265002240169849, 'R_rmse': 0.005671856791578847, 'T_mae': 0.0031010779785954783, 'T_rmse': 0.007921307126096637, 'phiR_mae': 0.024402686590994795, 'phiR_rmse': 0.10905509605919224, 'phiT_mae': 0.01827434982438468, 'phiT_rmse': 0.13273812912115077, 'R_o

({'label': 'am',
  'n': 5000,
  'R_mae': 0.0025169088971111007,
  'R_rmse': 0.005963946653039524,
  'T_mae': 0.0031418092457471784,
  'T_rmse': 0.007820708795250876,
  'phiR_mae': 0.02438346642204953,
  'phiR_rmse': 0.1074883793671806,
  'phiT_mae': 0.014347813825024636,
  'phiT_rmse': 0.11495134450534344,
  'R_out_of_01': 15,
  'T_out_of_01': 2},
 {'label': 'cr',
  'n': 5000,
  'R_mae': 0.00265002240169849,
  'R_rmse': 0.005671856791578847,
  'T_mae': 0.0031010779785954783,
  'T_rmse': 0.007921307126096637,
  'phiR_mae': 0.024402686590994795,
  'phiR_rmse': 0.10905509605919224,
  'phiT_mae': 0.01827434982438468,
  'phiT_rmse': 0.13273812912115077,
  'R_out_of_01': 17,
  'T_out_of_01': 1})

In [13]:
pos_test = np.array([9.81906933e+02, 8.67975732e+02, 9.41977049e+02, 9.27678994e+02,
 9.75982905e+02, 8.31890744e+02, 6.53923632e+02, 8.37959995e+02,
 9.21985370e+02, 7.74682001e+02, 9.93454774e+02, 7.25768577e+02,
 2.19316883e+02, 4.87712885e+02, 5.19326481e+02, 9.23125479e+02,
 4.85692989e+02, 5.12566172e+02, 7.41552996e+02, 7.16510746e+02,
 6.90332720e+02, 6.43673599e+02, 2.82473350e+02, 6.53451428e+01,
 3.17534948e+02, 1.56975911e+02, 5.98049070e+02, 3.64798669e+01,
 2.09424948e+02, 3.37403335e+02, 3.60524344e+02, 2.26015529e+02,
 3.01153404e+02, 4.72943603e+00, 6.25498466e+00, 8.00000000e-01,
 8.00000000e-01]
)

In [14]:
from pcm_pix.solution import save_preset

sol = save_preset(
    name="pos_test",
    pos=pos_test,
    cfg=CFG,
    sur0=sur0,
    sur1=sur1,
    preset_dir=CFG["preset_dir"],
    meta={"source": "test"},
)

In [15]:
"""
#Загрузить пресет для дальнейших расчётов/визуализации:
from pcm_pix.solution import load_preset

pos, cost, refl, trans, phi_refl, phi_trans, meta = load_preset("pos_test")

from pcm_pix.solution import Solution
from pathlib import Path
sol = Solution.from_json(Path(CFG["preset_dir"]) / "pos_test.json")
pos_m = sol.pos                       
R_am = sol.reflection["am"]
phi_R_am = sol.phase_shift_reflection["am"]
"""

'\n#Загрузить пресет для дальнейших расчётов/визуализации:\nfrom pcm_pix.solution import load_preset\n\npos, cost, refl, trans, phi_refl, phi_trans, meta = load_preset("pos_test")\n\nfrom pcm_pix.solution import Solution\nfrom pathlib import Path\nsol = Solution.from_json(Path(CFG["preset_dir"]) / "pos_test.json")\npos_m = sol.pos                       \nR_am = sol.reflection["am"]\nphi_R_am = sol.phase_shift_reflection["am"]\n'

In [16]:
#Проверить каталог пресетов, найти дубли и дополнить недостающие поля:
from pcm_pix.solution import check_and_fix_presets

report = check_and_fix_presets(
    cfg=CFG,
    sur0=sur0,
    sur1=sur1,
)

report

{'checked': 6, 'fixed': [], 'skipped_no_surrogates': [], 'duplicates': []}

In [17]:


# --- Presets / solutions catalog ---
from pcm_pix.solution import save_solution, load_solution

PRESETS = {
    "pos_2026_03_03_example": np.array([
        999.7153881712717, 989.8568611314538, 846.9306855895525, 812.3405141712682,
        357.50476722603656, 999.0189388651343, 853.5246592828234, 979.0165540829283,
        952.2077494627958, 998.2131906719599, 998.0020446893369, 638.3004244094977,
        645.2087664126163, 681.6157122198844, 702.1334382560799, 284.8875621006104,
        882.7549390819086, 520.3952331766168, 442.78456983718775, 875.627636396043,
        683.0421367034168, 725.5677666087739, 305.7428105275045, 302.3913087111928,
        246.68691437766532, 261.130662543832, 25.83901557205877, 397.619402446743,
        17.140592766392103, 140.12532478580698, 735.0979445623718, 211.69540821165236,
        355.55785007186574, 4.522892213428676, 2.827729567363648,
        0.9073471746931481, 0.8273584342733484
    ]),
    "pos_server_2026_03_04_night":np.array([
        9.99613315e+02, 8.86918846e+02, 6.06035395e+02, 8.54046947e+02,
        5.17645193e+02, 8.79810042e+02, 5.65420119e+02, 9.64383427e+02,
        9.97814618e+02, 9.55635573e+02, 9.95016420e+02, 5.99598526e+02,
        8.29227746e+02, 5.39970803e+02, 8.04025215e+02, 4.17264700e+02,
        6.15239459e+02, 3.62347946e+02, 4.20726459e+02, 6.09650053e+02,
        2.69023666e+02, 4.13247590e+02, 3.76852495e+02, 5.40484452e+02,
        1.99690690e+02, 3.57938384e+02, 4.35299951e+01, 2.57512612e+02,
        2.45768992e+02, 2.64399901e+02, 5.09649663e+02, 1.68530071e+02,
        2.33417540e+01, 4.73217432e+00, 3.35695242e+00, 9.56090488e-01,
        8.91457136e-01
    ]),
    "pos_test_2026_03_12": np.array([9.96841358e+02, 9.37302864e+02, 9.57294114e+02, 5.29040371e+02,
        9.77389395e+02, 8.58376681e+02, 8.42285256e+02, 9.33850847e+02,
        7.63770840e+02, 8.68523525e+02, 9.93136720e+02, 7.13942337e+02,
        1.72741142e+02, 4.06342614e+02, 3.83147986e+02, 6.38101254e+02,
        6.60311825e+02, 5.07564561e+02, 8.34434864e+02, 6.96096300e+02,
        6.80798657e+02, 6.45862244e+02, 2.87477815e+02, 7.04086071e+01,
        4.42449666e+01, 2.50460328e+02, 3.48349163e+02, 2.58011572e+02,
        7.10594600e+01, 3.19886408e+02, 2.33876373e+02, 2.56188765e+02,
        3.06062295e+02, 4.67935757e+00, 6.16194898e+00, 8.00000000e-01,
        8.00000000e-01]),
    "pos_server_2026_03_12": np.array([996.0443470708294, 572.8271927555644, 984.2487359207097, 887.5327249259412,
        881.9807993287004, 566.0181896913394, 876.0626657424411, 746.7439603540174, 
        804.525380579887, 901.4929955109988, 998.1095763354482, 724.4984270316959,
        106.6907199378395, 773.1027097258207, 477.17775717341044, 556.8891159329191,
        433.50739999462724, 577.3330032905027, 592.0556864456012, 711.9341242406858,
        685.0899094035707, 645.4252049086351, 318.94850557947257, 48.14530822881471, 
        574.9734674452097, 38.36816798416811, 161.8944738672979, 182.48062067122947, 
        199.78375691051463, 175.81728788584587, 272.8027577304986, 280.58320486862567, 
        312.024253921496, 4.6648036497638525, 0.0, 0.8, 
        0.8
  ]),

}

# save presets once (idempotent)
for name, pos in PRESETS.items():
    save_solution(run, name=name, pos=pos, cost=None, meta={"source": "preset"})

logger.info("presets saved to %s", run.results / "solutions")




2026-03-14 16:06:34,191 | INFO | saved solution pos_2026_03_03_example.json
2026-03-14 16:06:34,193 | INFO | saved solution pos_server_2026_03_04_night.json
2026-03-14 16:06:34,195 | INFO | saved solution pos_test_2026_03_12.json
2026-03-14 16:06:34,197 | INFO | saved solution pos_server_2026_03_12.json
2026-03-14 16:06:34,198 | INFO | presets saved to outputs/2026_03_12/results/solutions


In [25]:
from pcm_pix.hyperopt2 import resolve_hyperparams

resolve_hyperparams("pso", sur0, sur1, CFG, run, logger)

pos0 = PRESETS[CFG["preset_name"]]
resolve_hyperparams("de", sur0, sur1, CFG, run, logger, pos0=pos0)


2026-03-16 12:29:45,283 | INFO | PSO hyperparams loaded from pso_refine.csv: c1=1.365595582786494 c2=1.3004900273877351 w=0.3332738612391048
2026-03-16 12:29:45,288 | INFO | DE hyperparams loaded from de_refine.csv: mutation=0.2420622190083343 recombination=0.7001234155368052 popsize=54


DEHyperParams(mutation=0.2420622190083343, recombination=0.7001234155368052, popsize=54, source='de_refine.csv')

In [26]:
from pcm_pix.optimize import load_hyperopt_params
loaded = load_hyperopt_params(CFG, run=run)
logger.info("hyperopt params loaded: %s", loaded)

2026-03-16 12:57:20,777 | INFO | No hyperparams file found: data/hyperparams_final.json
2026-03-16 12:57:20,778 | INFO | hyperopt params loaded: False


In [ ]:
#CFG['de_mutation'] = 0.99
#CFG['de_recombination'] = 0.1 
#CFG['de_popsize'] = 60

In [ ]:
import numpy as np
import torch

from pcm_pix.optimize import (
    f_vec,
    run_pso,
    run_de_full,
    run_hybrid_pso_de,
)


def run_one(seed: int):
    # фиксируем сида для всего, что можем
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    logger.info("=== RUN seed=%s ===", seed)

    # тут ровно твой блок оптимизации, только возвращаем (pos, cost)
    if CFG["opt_mode"] == "preset":
        pos = PRESETS[CFG["preset_name"]]
        cost = float(f_vec(pos.reshape(1, -1), sur0, sur1, CFG)[0])
        logger.info("using preset %s cost=%.6f", CFG["preset_name"], cost)

    elif CFG["opt_mode"] == "pso":
        best = run_pso(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    elif CFG["opt_mode"] == "de_full":
        pos0 = PRESETS[CFG["preset_name"]]
        best = run_de_full(sur0, sur1, CFG, run, pos=pos0)
        pos, cost = best.pos, best.cost

    else:
        best = run_hybrid_pso_de(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    logger.info("FINAL cost=%.6f", float(cost))
    logger.info("FINAL pos_len=%s", len(pos))

    return np.array(pos), float(cost)

In [22]:
K = 50  # сколько прогонов
base_seed = 100  # базовый сид, можно взять из CFG или руками

best_cost = np.inf
best_pos = None
best_seed = None

for k in range(K):
    seed = base_seed + k
    pos_k, cost_k = run_one(seed)

    logger.info("RUN %s/%s seed=%s cost=%.6f", k + 1, K, seed, cost_k)

    if cost_k < best_cost:
        best_cost = cost_k
        best_pos = pos_k
        best_seed = seed

logger.info("=== BEST over %s runs: seed=%s cost=%.6f ===", K, best_seed, best_cost)
best_cost, best_seed

2026-03-14 16:06:35,573 | INFO | === RUN seed=100 ===
2026-03-14 16:06:35,607 | INFO | PSO start: particles=3000 iters=1000 dims=37
2026-03-14 16:06:35,608 - pyswarms.single.global_best - INFO - Optimize for 1000 iters with {'c1': 1.365595582786494, 'c2': 1.3004900273877351, 'w': 0.3332738612391048}
pyswarms.single.global_best:   0%|          |0/1000

pyswarms.single.global_best: 100%|██████████|1000/1000, best_cost=2.11 
2026-03-14 16:08:14,536 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 2.1126537270094676, best pos: [6.02429027e-07 6.48031553e-07 4.65497610e-07 6.08222257e-07
 6.41183311e-07 8.07784606e-07 9.66760613e-07 6.53261059e-07
 8.94962189e-07 8.61803553e-07 9.80021992e-07 4.99489413e-07
 4.77281457e-07 3.10894695e-07 3.53748175e-07 4.24830391e-07
 6.81164803e-07 6.42211676e-07 4.76077312e-07 6.55611852e-07
 5.88562379e-07 6.74687554e-07 8.56831637e-08 8.76499425e-08
 4.09863595e-08 4.53252314e-08 4.73772783e-08 5.02943968e-07
 3.42099159e-07 4.83403181e-08 2.60947275e-07 1.51628128e-07
 2.97276718e-07 2.06362505e+00 1.83477265e+00 8.00064695e-01
 8.44426894e-01]
2026-03-14 16:08:14,538 | INFO | PSO done: cost=2.1126537270094676
2026-03-14 16:08:14,679 | INFO | DE start (full): maxiter=50000 popsize=60 init_mode=init_ar lc=False
2026-03-14 16:08:16,684 | INFO | Conv 0 2.105043
2026-03-14 16:08

KeyboardInterrupt: 

In [19]:
# --- Save final solution to catalog ---
from pcm_pix.optimize import save_solution

name = f"final_{CFG['opt_mode']}"
path = save_solution(run, name=name, pos=best_pos, cost=float(best_cost), meta={"cfg": {"opt_mode": CFG["opt_mode"]}})
logger.info("final solution saved: %s", path.name)
path

2026-03-12 15:13:45,959 | INFO | saved solution final_hybrid_pso_de.json
2026-03-12 15:13:45,962 | INFO | final solution saved: final_hybrid_pso_de.json


PosixPath('outputs/2026_03_12/results/solutions/final_hybrid_pso_de.json')